In [1]:
import kagglehub
import pandas as pd
import numpy as np

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
path = kagglehub.dataset_download("prokshitha/home-value-insights")
houses = pd.read_csv(path + '/house_price_regression_dataset.csv')

In [10]:
houses

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
0,1360,2,1,1981,0.599637,0,5,2.623829e+05
1,4272,3,3,2016,4.753014,1,6,9.852609e+05
2,3592,1,2,2016,3.634823,0,9,7.779774e+05
3,966,1,2,1977,2.730667,1,8,2.296989e+05
4,4926,2,1,1993,4.699073,0,8,1.041741e+06
...,...,...,...,...,...,...,...,...
995,3261,4,1,1978,2.165110,2,10,7.014940e+05
996,3179,1,2,1999,2.977123,1,10,6.837232e+05
997,2606,4,2,1962,4.055067,0,2,5.720240e+05
998,4723,5,2,1950,1.930921,0,7,9.648653e+05


In [11]:
houses.isna().sum()
columns = houses.columns

In [12]:
houses=pd.DataFrame(data=houses, columns=columns)
X = houses.drop('House_Price', axis=1)
y = houses['House_Price']

In [13]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(x_train, y_train)
y_pred = model.predict(x_test)

In [15]:
from sklearn.metrics import mean_squared_error, r2_score
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Output results
print("Coefficients:", model.coef_)
print("Mean Squared Error:", mse)
print("R-squared Score:", r2)

Coefficients: [  199.5132451  10225.20442447  8208.43477773   993.53717106
 14885.38441462  5146.14838284   115.06859524]
Mean Squared Error: 101434798.50566797
R-squared Score: 0.9984263636823408


In [16]:
y_pred_train = model.predict(x_train)
mse_train = mean_squared_error(y_train, y_pred_train)
r2_train = r2_score(y_train, y_pred_train)

print("Mean Squared Error:", mse_train)
print("R-squared Score:", r2_train)

Mean Squared Error: 93850531.95601234
R-squared Score: 0.9985375946918145


In [19]:
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

mlflow.set_tracking_uri("http://mlflow:5000/")
with mlflow.start_run():
    ## Parametrii modelului / antrenarii
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)

    ## Metrici pe setul de test
    mlflow.log_metric("mse_test", mse)
    mlflow.log_metric("r2_test", r2)

    ## Metrici pe setul de train
    mlflow.log_metric("mse_train", mse_train)
    mlflow.log_metric("r2_train", r2_train)

    ## Schema (signature) - ce input primeste modelul si ce output produce
    signature = infer_signature(x_train, y_pred_train)
    
    ## Exemplu de input, util cand testezi morelul direct din UI-ul de MLflow
    input_example = x_train.iloc[:5]

    ## Log model + inregistrare directa in Model Registry
    mlflow.sklearn.log_model(
        sk_model = model,
        artifact_path="model",
        signature = signature,
        input_example=input_example,
        registered_model_name="HousePriceRegressor"
    )   

/usr/local/lib/python3.11/site-packages/mlflow/types/utils.py:393: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Successfully registered model 'HousePriceRegressor'.
2026/08/28 16:27:01 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: HousePriceRegressor, version 1
Created version '1'

In [21]:
from sklearn.linear_model import Ridge

mlflow.set_tracking_uri("http://mlflow:5000/")
mlflow.set_experiment("Regression Experiment")

configs = [
    {"alpha": 0.1,},
    {"alpha": 1.0},
    {"alpha": 10.0}
]

for i, params in enumerate(configs):
    with mlflow.start_run():
        model = Ridge(**params)
        model.fit(x_train, y_train)

        y_pred_test = model.predict(x_test)
        y_pred_train = model.predict(x_train)

        mse_test = mean_squared_error(y_test, y_pred_test)
        r2_test = r2_score(y_test, y_pred_test)

        mse_train = mean_squared_error(y_train, y_pred_train)
        r2_train = r2_score(y_train, y_pred_train)
        
        ## Parametrii modelului / antrenarii
        mlflow.log_param("model_type", "Ridge")
        mlflow.log_params(params)
    
        ## Metrici pe setul de test
        mlflow.log_metric("mse_test", mse)
        mlflow.log_metric("r2_test", r2)
    
        ## Metrici pe setul de train
        mlflow.log_metric("mse_train", mse_train)
        mlflow.log_metric("r2_train", r2_train)
    
        ## Schema (signature) - ce input primeste modelul si ce output produce
        signature = infer_signature(x_train, y_pred_train)
        
        ## Exemplu de input, util cand testezi morelul direct din UI-ul de MLflow
        input_example = x_train.iloc[:5]
    
        ## Log model + inregistrare directa in Model Registry
        mlflow.sklearn.log_model(
            sk_model = model,
            artifact_path="model",
            signature = signature,
            input_example=input_example
        )   

/usr/local/lib/python3.11/site-packages/mlflow/types/utils.py:393: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/_distutils_hack/__init__.py:15: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or